# 3 — Model: CNN Architecture

**Sprint 3** — Model Definition

This notebook defines the CNN model used to classify audio genres from mel spectrogram tensors.

- Re-instantiates the train/val/test DataLoaders from `2_transform.ipynb` using the same `load_split()` + `AudioDataset` pattern
- Defines `AudioCNN`, a 3-block convolutional network in PyTorch
- Verifies the forward pass produces the expected output shape `[batch, 10]`
- Instantiates the loss function (`CrossEntropyLoss`) and optimizer (`Adam`)

**Input shape:** `[batch, 1, 128, 1292]` — (batch, mono channel, mel bins, time frames)  
**Output shape:** `[batch, 10]` — one logit per genre class

## 1. Imports and Configuration

All configuration constants are kept identical to `2_transform.ipynb` so that the DataLoaders and mel spectrogram pipeline behave exactly as they did during preprocessing. `torch.nn` is added here for model definition.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("NUMBA_CACHE_DIR", str(Path.cwd() / ".numba-cache"))

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


RANDOM_SEED = 42
SAMPLE_RATE = 22_050
CLIP_DURATION_SECONDS = 30
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
BATCH_SIZE = 16
FREQ_MASK_MAX_WIDTH = 16

FIXED_NUM_SAMPLES = SAMPLE_RATE * CLIP_DURATION_SECONDS

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2. DataLoaders

The DataLoaders are re-instantiated here by replicating the `load_split()` + `AudioDataset` pattern from `2_transform.ipynb`. Notebooks do not share runtime state, so all helper functions and the DB connection must be redefined. No logic is changed — this is a straight re-use of the preprocessing pipeline.

In [ ]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "code").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the audio-genre-classifier repo root.")


repo_root = find_repo_root(Path.cwd())
load_dotenv(repo_root / ".env")

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_HOST     = "localhost"
DB_PORT     = 5432
DB_NAME     = "audio_genre_classifier"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

try:
    with engine.connect() as con:
        db_name = con.execute(text("SELECT current_database()")).fetchone()[0]
    print(f"Connected to: {db_name}")
except Exception as e:
    print(f"Connection failed: {e}")

In [ ]:
def load_split(split_name: str, engine) -> pd.DataFrame:
    """Load usable (non-corrupted, non-duplicate) songs for one split."""
    assert split_name in ("train", "val", "test"), \
        f"split_name must be 'train', 'val', or 'test' — got '{split_name}'"
    query = text("""
        SELECT ac.file_path, l.label_name AS label, ac.split
        FROM   audio_clips ac
        JOIN   labels l ON l.label_id = ac.label_id
        WHERE  ac.split        = :split_name
          AND  ac.is_corrupted = FALSE
          AND  ac.is_duplicate = FALSE
        ORDER  BY ac.file_path
    """)
    df = pd.read_sql(sql=query, con=engine, params={"split_name": split_name})
    df["file_path"] = df["file_path"].apply(lambda p: str(repo_root / p))
    return df


def pad_or_truncate(audio: np.ndarray, target_num_samples: int) -> np.ndarray:
    if len(audio) > target_num_samples:
        return audio[:target_num_samples]
    if len(audio) < target_num_samples:
        return np.pad(audio, (0, target_num_samples - len(audio)), mode="constant")
    return audio


def augment_waveform(audio: np.ndarray) -> np.ndarray:
    gain = np.random.uniform(0.8, 1.2)
    audio = audio * gain
    max_shift = int(0.1 * SAMPLE_RATE)
    shift = np.random.randint(-max_shift, max_shift + 1)
    audio = np.roll(audio, shift)
    noise = np.random.normal(0, 0.005, size=audio.shape)
    return (audio + noise).astype(np.float32)


def audio_to_mel_spectrogram(audio: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS, power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db.astype(np.float32)


def augment_mel_spectrogram(mel: np.ndarray) -> np.ndarray:
    mel = mel.copy()
    max_width = min(FREQ_MASK_MAX_WIDTH, mel.shape[0])
    mask_width = np.random.randint(0, max_width + 1)
    if mask_width > 0:
        start = np.random.randint(0, mel.shape[0] - mask_width + 1)
        mel[start:start + mask_width, :] = 0.0
    return mel.astype(np.float32)


class AudioDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        label_to_idx: dict[str, int],
        sample_rate: int = SAMPLE_RATE,
        fixed_num_samples: int = FIXED_NUM_SAMPLES,
        augment: bool = False,
    ):
        expected_columns = {"file_path", "label", "split"}
        missing_columns = expected_columns - set(dataframe.columns)
        if missing_columns:
            raise ValueError(f"Dataframe is missing columns: {sorted(missing_columns)}")
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.label_to_idx = label_to_idx
        self.sample_rate = sample_rate
        self.fixed_num_samples = fixed_num_samples
        self.augment = augment

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.dataframe.iloc[index]
        audio, _ = librosa.load(row["file_path"], sr=self.sample_rate, mono=True)
        audio = pad_or_truncate(audio, self.fixed_num_samples)
        if self.augment and row["split"] == "train":
            audio = augment_waveform(audio)
        mel = audio_to_mel_spectrogram(audio)
        if self.augment and row["split"] == "train":
            mel = augment_mel_spectrogram(mel)
        features = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)
        target = torch.tensor(self.label_to_idx[row["label"]], dtype=torch.long)
        return features, target

In [ ]:
metadata_df = pd.read_sql(
    sql=text("""
        SELECT ac.file_path, l.label_name AS label, ac.split
        FROM   audio_clips ac
        JOIN   labels l ON l.label_id = ac.label_id
        WHERE  ac.is_corrupted = FALSE
          AND  ac.is_duplicate = FALSE
        ORDER  BY ac.file_path
    """),
    con=engine,
)

labels = sorted(metadata_df["label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

train_df = load_split("train", engine)
val_df   = load_split("val",   engine)
test_df  = load_split("test",  engine)

train_dataset = AudioDataset(train_df, label_to_idx=label_to_idx, augment=True)
val_dataset   = AudioDataset(val_df,   label_to_idx=label_to_idx, augment=False)
test_dataset  = AudioDataset(test_df,  label_to_idx=label_to_idx, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train examples : {len(train_dataset)}")
print(f"Val examples   : {len(val_dataset)}")
print(f"Test examples  : {len(test_dataset)}")
print(f"Label mapping  : {label_to_idx}")

## 3. CNN Architecture

`AudioCNN` treats each mel spectrogram as a single-channel image and learns hierarchical frequency-time patterns through three convolutional blocks.

**Block structure — Conv2d → BatchNorm2d → ReLU → MaxPool2d:**

| Block | In channels | Out channels | Spatial size after pool |
|-------|------------|--------------|-------------------------|
| 1     | 1          | 32           | [32, 64, 646]           |
| 2     | 32         | 64           | [64, 32, 323]           |
| 3     | 64         | 128          | [128, 16, 161]          |

**Architecture decisions:**

- **3×3 convolutions with `padding=1`** — preserves spatial dimensions before pooling so each block halves H and W cleanly via `MaxPool2d(2, 2)`. 3×3 is the standard kernel for capturing local frequency-time patterns.
- **Doubling channels per block (32→64→128)** — compensates for the spatial information lost at each pooling step so the total feature volume stays roughly constant. No single layer becomes a bottleneck.
- **BatchNorm2d after every Conv2d** — normalizes activations before the non-linearity, which stabilizes gradients and allows a higher learning rate without careful initialization.
- **`AdaptiveAvgPool2d((4, 4))` before the classifier** — reduces `[128, 16, 161]` to a fixed `[128, 4, 4]` (2,048 values). Flattening directly after block 3 would produce a 329,728-element vector and a ~84M-parameter FC layer on a 700-example training set. The adaptive pool keeps the FC layer at a manageable 2,048→256 (~500K params) and adds mild translation invariance.
- **Dropout(0.5)** — applied after the hidden FC layer, the widest point of the classifier head and where overfitting risk is highest on a small dataset.
- **10 output units, no final activation** — `CrossEntropyLoss` applies log-softmax internally; adding a softmax here would double-apply it.

In [ ]:
class AudioCNN(nn.Module):
    """3-block CNN for mel spectrogram genre classification."""

    def __init__(self, num_classes: int = 10):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Reduces [128, 16, 161] → [128, 4, 4] regardless of input time-axis length.
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.adaptive_pool(x)
        x = self.classifier(x)
        return x

## 4. Device, Model Instantiation, and Parameter Count

The model is moved to GPU if one is available; otherwise it runs on CPU. Printing the total parameter count confirms the model loaded correctly and gives a quick sanity check on architecture size before any training.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = AudioCNN(num_classes=10).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

## 5. Loss Function and Optimizer

**`CrossEntropyLoss`** — the standard loss for multi-class classification. It combines `LogSoftmax` and `NLLLoss` internally, so the output layer produces raw logits (no activation needed). GTZAN is balanced across 10 classes so no class weights are applied.

**`Adam` with `lr=1e-3`** — Adam adapts the learning rate per parameter using first and second moment estimates, making it less sensitive to the initial learning rate than SGD. `1e-3` is the conventional Adam starting point for small-to-medium CNN tasks and pairs well with BatchNorm.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"Loss      : {criterion}")
print(f"Optimizer : {optimizer.__class__.__name__}, lr={optimizer.param_groups[0]['lr']}")

## 6. Forward Pass Verification

Runs one batch through the model to confirm the output shape is `[batch, 10]` and that no errors occur. This catches shape mismatches or device mismatches before a full training run that could take hours.

In [ ]:
model.eval()
with torch.no_grad():
    sample_features, sample_labels = next(iter(train_loader))
    sample_features = sample_features.to(device)
    sample_labels   = sample_labels.to(device)

    output = model(sample_features)

print(f"Input shape  : {sample_features.shape}")
print(f"Output shape : {output.shape}")
print(f"Expected     : [{sample_features.shape[0]}, 10]")
assert output.shape == (sample_features.shape[0], 10), "Output shape mismatch!"
print("Forward pass OK")

## Handoff Notes — What the Training Notebook Needs to Know

**What this notebook leaves ready:**

- `AudioCNN` — the model class, fully defined and verified. Swap this in wherever the training notebook uses a placeholder.
- `train_loader`, `val_loader`, `test_loader` — DataLoaders pulling real audio from PostgreSQL, batch size 16, mel spectrograms of shape `[1, 128, 1292]`.
- `criterion` — `CrossEntropyLoss`, no class weights needed (dataset is balanced).
- `optimizer` — `Adam`, `lr=1e-3`, attached to the model parameters.
- `device` — set to `cuda` if available, else `cpu`. Move all tensors and the model to this before training.

**Key numbers to know:**

| | Value |
|---|---|
| Trainable parameters | 620,234 |
| Batch size | 16 |
| Train / Val / Test | 699 / 150 / 150 songs |
| Input tensor shape | `[16, 1, 128, 1292]` |
| Output tensor shape | `[16, 10]` |

**Watch out for overfitting** — 620K parameters on 699 training examples is a tight ratio. Monitor val loss each epoch and save the best checkpoint by val accuracy, not final epoch. Early stopping is worth considering if val loss starts climbing while train loss keeps falling.

--------------

## Summary

This notebook defines the CNN model for GTZAN genre classification.

**What was done:**
- Re-instantiated train/val/test DataLoaders using the same `load_split()` + `AudioDataset` pipeline from `2_transform.ipynb`
- Defined `AudioCNN`: 3 convolutional blocks (Conv2d → BatchNorm2d → ReLU → MaxPool2d), adaptive average pooling, and a fully-connected classifier head with Dropout
- Confirmed forward pass produces `[batch, 10]` output
- Instantiated `CrossEntropyLoss` and `Adam(lr=1e-3)`

**Architecture summary:**

| Component | Details |
|---|---|
| Input | `[batch, 1, 128, 1292]` |
| Block 1 | Conv2d(1→32, 3×3) → BN → ReLU → MaxPool |
| Block 2 | Conv2d(32→64, 3×3) → BN → ReLU → MaxPool |
| Block 3 | Conv2d(64→128, 3×3) → BN → ReLU → MaxPool |
| Pool | AdaptiveAvgPool2d → `[128, 4, 4]` |
| Classifier | Flatten → Linear(2048, 256) → ReLU → Dropout(0.5) → Linear(256, 10) |
| Output | `[batch, 10]` logits |
| Loss | CrossEntropyLoss |
| Optimizer | Adam, lr=1e-3 |